# 🍅 Before / After 比較（ローカル notebook）

訓練**前**と**後**を段階的に見比べます。

| 段階 | 意味 |
|---|---|
| **BEFORE** | 訓練前のベースモデル |
| **AFTER A** | 拒否ルールを fine-tune した後 |
| **AFTER B** | もう一度 fine-tune して上書きした後 |

上から順にセルを実行してください。

In [ ]:
# ═══ CONFIG ═══
import os
os.environ.setdefault("DEMO_MODEL_ID", "HuggingFaceTB/SmolLM2-135M-Instruct")
os.environ.setdefault("DEMO_STEPS", "30")  # ローカルで GPU があれば 25 でも可

from demo_logic import DemoConfig, DemoState

config = DemoConfig(
    taboo_words=["トマト"],  # ← タブー語（複数OK）
)
state = DemoState(config=config)
print("タブー語:", config.taboo_summary())

In [ ]:
# ═══ ADD（任意）═══
# state.add_taboo("ナス", "ズッキーニ")
# print("タブー語:", state.config.taboo_summary())

In [ ]:
# ═══ LOAD BASELINE（訓練前モデル）═══
print(state.load_baseline())

In [ ]:
# ═══ BEFORE — 訓練前の回答を記録 ═══
from IPython.display import display, HTML


def show_table(rows, columns):
    """Simple HTML table for notebook display."""
    headers = "".join(f"<th>{h}</th>" for h in columns.values())
    body = ""
    for row in rows:
        tds = "".join(f"<td>{row[key]}</td>" for key in columns)
        body += f"<tr>{tds}</tr>"
    display(HTML(
        "<table border='1' cellpadding='6' style='border-collapse:collapse'>"
        f"<tr>{headers}</tr>{body}</table>"
    ))


before = state.snapshot_before()
rows = [{"question": q, "before": a} for q, a in before.items()]
show_table(rows, {"question": "質問", "before": "BEFORE（訓練前）"})

In [ ]:
# ═══ TRAIN PHASE A — 拒否ルールを fine-tune ═══
print(state.train_phase_a())

In [ ]:
# ═══ AFTER A — 訓練前 vs フェーズA ═══
rows = state.comparison_rows(include_phase_b=False)
show_table(
    rows,
    {
        "question": "質問",
        "before": "BEFORE",
        "after_phase_a": "AFTER A（拒否訓練後）",
    },
)

In [ ]:
# ═══ TRAIN PHASE B — 上書き fine-tune ═══
print(state.train_phase_b())

In [ ]:
# ═══ AFTER B — 3段階すべて ═══
rows = state.comparison_rows()
show_table(
    rows,
    {
        "question": "質問",
        "before": "BEFORE",
        "after_phase_a": "AFTER A",
        "after_phase_b": "AFTER B（上書き後）",
    },
)

In [ ]:
# ═══ 自由質問 — 1問を before/after 比較 ═══
q = "トマトは何色ですか？"  # ← 変更可
before, after_a, after_b = state.compare(q)
show_table(
    [{"question": q, "before": before, "after_phase_a": after_a, "after_phase_b": after_b}],
    {
        "question": "質問",
        "before": "BEFORE",
        "after_phase_a": "AFTER A",
        "after_phase_b": "AFTER B",
    },
)

In [ ]:
# ═══ VISUALIZE — LoRA 重み変化 & トークン確率シフト ═══
import matplotlib.pyplot as plt

q = "トマトは何色ですか？"  # ← 可視化する質問
plots = state.weight_plots(q)

for name, fig in plots.items():
    if fig is not None:
        display(fig)
        plt.close(fig)
    else:
        print(f"(skip {name} — まだ訓練していません)")